In [ ]:
from plotnine import *
import pandas as pd
import plotly.express as px
import numpy as np

perf_nlmixr = pd.read_csv("outputs/performance_nlmixr.csv")
perf_nlmixr["algo"] = "nlmixr"
perf_pysaem = pd.read_csv("outputs/performance_pysaem.csv")
perf_pysaem_full = perf_pysaem[["time", "nb_patients"]]
perf_pysaem_full["algo"] = "pysaem (train + SAEM)"
perf_pysaem_optim = perf_pysaem[["time_saem", "nb_patients"]].rename(
    columns={"time_saem": "time"}
)
perf_pysaem_optim["algo"] = "pysaem (SAEM only)"

full_df = pd.concat([perf_nlmixr, perf_pysaem_full, perf_pysaem_optim])

In [ ]:
display(full_df)

In [ ]:
fig = px.scatter(
    full_df,
    y="time",
    x="nb_patients",
    color="algo",
    log_x=True,
    log_y=True,
    title="Run time for SAEM on TMDD model<br>(100/100 iterations, 1 chain, 5x) depending on observed data set size",
    labels={
        "time": "Runtime (s)",
        "nb_patients": "Number of patients",
        "algo": "Implementation",
    },
    template="ggplot2",
    trendline="ols",
    trendline_options=dict(log_x=True, log_y=True),
    opacity=0.5,
)
fig.update_traces(marker=dict(size=10))
fig.update_layout(
    boxgap=0.0,
    boxgroupgap=0.0,
    width=800,
    height=400,
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.3),
    margin=dict(t=50, l=30, r=30, b=10),
)
fig.show()

In [ ]:
ebe_pysaem = pd.read_csv("outputs/ebe_pysaem.csv")
ebe_pysaem["algo"] = "pysaem"
ebe_nlmixr = pd.read_csv("outputs/ebe_nlmixr.csv")
ebe_nlmixr["algo"] = "nlmixr"

ebe_df = pd.concat([ebe_pysaem, ebe_nlmixr])

descriptors = ["k_eL", "R0", "Vc"]

ebe_df = ebe_df.melt(
    id_vars=["id", "algo"], value_vars=descriptors, var_name="descriptor"
).reset_index()
display(ebe_df.head())


ebe_true = (
    pd.read_csv("data/obs_data_200.csv")[["id", "true_k_eL", "true_R0", "true_Vc"]]
    .drop_duplicates()
    .rename(columns={"true_k_eL": "k_eL", "true_R0": "R0", "true_Vc": "Vc"})
    .melt(
        id_vars=["id"],
        value_vars=["k_eL", "R0", "Vc"],
        var_name="descriptor",
        value_name="true_value",
    )
)

err_df = ebe_df.merge(ebe_true, on=["id", "descriptor"])
err_df["abs_error"] = err_df["value"] - err_df["true_value"]
err_df["rel_error"] = err_df["abs_error"] / err_df["true_value"]
display(err_df.head())

comp_df = pd.concat(
    [ebe_df, ebe_true.rename(columns={"true_value": "value"}).assign(algo="true")]
)
display(comp_df.head())

In [ ]:
(
    ggplot(err_df, aes(x="true_value", y="rel_error", color="algo"))
    + geom_point()
    + facet_wrap("descriptor", scales="free")
    + scale_x_continuous(trans="log10")
)

In [ ]:
(
    ggplot(comp_df, aes(x="value", fill="algo", color="algo"))
    + geom_density(alpha=0.3)
    + facet_wrap("descriptor", scales="free")
    + scale_x_continuous(trans="log10")
    + theme_minimal()
    + theme(figure_size=(10, 4))
)